# HeartGuard — Data Exploration

This notebook performs exploratory data analysis on the Cleveland Heart Disease dataset.

**Status:** Phase 3 — Data Pipeline & Exploratory Data Analysis

**Note:** No models are trained in this notebook. All visualizations use actual dataset values.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent if not Path().resolve().name == "HeartGuard" else Path()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import (
    load_cleveland_dataset,
    load_framingham_dataset,
    get_dataset_info,
    normalize_cleveland_target,
)
from src.data.preprocessing import (
    get_missing_value_report,
    get_duplicate_report,
)
from src.data.quality import generate_data_quality_report, print_dataset_summary
from src.data.features import (
    TARGET_COLUMN,
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
)
from src.utils.validators import check_invalid_values, detect_outliers_iqr

print("Libraries loaded successfully.")

## Section 1 — Load Cleveland Dataset

In [ ]:
try:
    cleveland = load_cleveland_dataset()
    cleveland = normalize_cleveland_target(cleveland)
    print(f"Cleveland dataset loaded: {cleveland.shape[0]} rows x {cleveland.shape[1]} columns")
except FileNotFoundError as e:
    print(f"Cleveland dataset not found: {e}")
    print("Please place heart.csv in data/raw/ and rerun this notebook.")
    cleveland = None

## Section 2 — Dataset Dimensions

In [ ]:
if cleveland is not None:
    print(f"Shape: {cleveland.shape}")
    print(f"Rows: {cleveland.shape[0]}")
    print(f"Columns: {cleveland.shape[1]}")
    print(f"\nColumn names: {list(cleveland.columns)}")
    cleveland.head(10)

## Section 3 — Column Information & Data Types

In [ ]:
if cleveland is not None:
    print("Data types:")
    print(cleveland.dtypes)
    print(f"\nNumerical features: {[c for c in NUMERICAL_FEATURES if c in cleveland.columns]}")
    print(f"Categorical features: {[c for c in CATEGORICAL_FEATURES if c in cleveland.columns]}")

## Section 4 — Missing Values

In [ ]:
if cleveland is not None:
    missing_report = get_missing_value_report(cleveland)
    print(missing_report.to_string(index=False))
    print(f"\nTotal missing values: {cleveland.isnull().sum().sum()}")

## Section 5 — Duplicate Records

In [ ]:
if cleveland is not None:
    dup_report = get_duplicate_report(cleveland)
    for key, value in dup_report.items():
        print(f"  {key}: {value}")

## Section 6 — Invalid Values

In [ ]:
if cleveland is not None:
    invalid = check_invalid_values(cleveland)
    if invalid:
        for col, indices in invalid.items():
            print(f"  {col}: {len(indices)} invalid values at rows {indices[:10]}")
    else:
        print("No invalid values detected.")

## Section 7 — Target Distribution

In [ ]:
if cleveland is not None and TARGET_COLUMN in cleveland.columns:
    target_counts = cleveland[TARGET_COLUMN].value_counts().sort_index()
    print("Target distribution (0=absence, 1=presence):")
    for val, count in target_counts.items():
        pct = count / len(cleveland) * 100
        print(f"  {val}: {count} ({pct:.1f}%)")

    fig, ax = plt.subplots(figsize=(6, 4))
    target_counts.plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"])
    ax.set_title("Target Distribution")
    ax.set_xlabel("Heart Disease")
    ax.set_ylabel("Count")
    ax.set_xticklabels(["Absence (0)", "Presence (1)"], rotation=0)
    plt.tight_layout()
    plt.show()

## Section 8 — Numerical Feature Statistics

In [ ]:
if cleveland is not None:
    cleveland.describe().round(2)

## Section 9 — Categorical Feature Distributions

In [ ]:
if cleveland is not None:
    for col in CATEGORICAL_FEATURES:
        if col in cleveland.columns:
            print(f"\n{col}:")
            print(cleveland[col].value_counts().sort_index())

## Section 10 — Numerical Feature Distributions

In [ ]:
if cleveland is not None:
    num_plot_cols = ["age", "resting_bp", "cholesterol", "max_heart_rate", "st_depression"]
    num_plot_cols = [c for c in num_plot_cols if c in cleveland.columns]
    n = len(num_plot_cols)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    colors = ["#3498db", "#9b59b6", "#e67e22", "#1abc9c", "#e74c3c"]
    for ax, col, color in zip(axes, num_plot_cols, colors):
        cleveland[col].hist(bins=30, ax=ax, color=color, edgecolor="black")
        ax.set_title(col)
        ax.set_xlabel(col)
        ax.set_ylabel("Frequency")
    plt.tight_layout()
    plt.show()

## Section 11 — Correlation Analysis

In [ ]:
if cleveland is not None:
    num_cols = [c for c in NUMERICAL_FEATURES if c in cleveland.columns]
    if TARGET_COLUMN in cleveland.columns:
        num_cols_with_target = num_cols + [TARGET_COLUMN]
    else:
        num_cols_with_target = num_cols
    corr = cleveland[num_cols_with_target].corr()

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(num_cols_with_target)))
    ax.set_yticks(range(len(num_cols_with_target)))
    ax.set_xticklabels(num_cols_with_target, rotation=45, ha="right")
    ax.set_yticklabels(num_cols_with_target)
    plt.colorbar(im, ax=ax)
    ax.set_title("Correlation Matrix — Numerical Features")
    plt.tight_layout()
    plt.show()

    print("Correlation with target:")
    if TARGET_COLUMN in corr.columns:
        print(corr[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values(ascending=False).round(3))

## Section 12 — Outlier Analysis (IQR)

In [ ]:
if cleveland is not None:
    num_cols = [c for c in NUMERICAL_FEATURES if c in cleveland.columns]
    outliers = detect_outliers_iqr(cleveland, columns=num_cols)
    if outliers:
        print("Potential outliers detected (IQR method, factor=1.5):\n")
        for col, info in outliers.items():
            print(f"  {col}:")
            print(f"    Count: {info['count']}")
            print(f"    IQR bounds: [{info['lower_bound']:.2f}, {info['upper_bound']:.2f}]")
            print(f"    Q1={info['q1']:.2f}, Q3={info['q3']:.2f}, IQR={info['iqr']:.2f}")
    else:
        print("No outliers detected.")
    print("\nNote: Medical datasets can legitimately contain extreme observations.")
    print("Outliers are flagged for analysis, not automatically removed.")

## Section 13 — Important Feature vs Target Visualizations

In [ ]:
if cleveland is not None and TARGET_COLUMN in cleveland.columns:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # Age vs Target
    for label, color in [(0, "#2ecc71"), (1, "#e74c3c")]:
        subset = cleveland[cleveland[TARGET_COLUMN] == label]
        axes[0, 0].hist(subset["age"], bins=20, alpha=0.6, color=color,
                        label=f"{'Absence' if label == 0 else 'Presence'}", edgecolor="black")
    axes[0, 0].set_title("Age by Target")
    axes[0, 0].set_xlabel("Age")
    axes[0, 0].legend()

    # Cholesterol vs Target
    for label, color in [(0, "#2ecc71"), (1, "#e74c3c")]:
        subset = cleveland[cleveland[TARGET_COLUMN] == label]
        axes[0, 1].hist(subset["cholesterol"], bins=20, alpha=0.6, color=color,
                        label=f"{'Absence' if label == 0 else 'Presence'}", edgecolor="black")
    axes[0, 1].set_title("Cholesterol by Target")
    axes[0, 1].set_xlabel("Cholesterol (mg/dl)")
    axes[0, 1].legend()

    # Max Heart Rate vs Target
    for label, color in [(0, "#2ecc71"), (1, "#e74c3c")]:
        subset = cleveland[cleveland[TARGET_COLUMN] == label]
        axes[1, 0].hist(subset["max_heart_rate"], bins=20, alpha=0.6, color=color,
                        label=f"{'Absence' if label == 0 else 'Presence'}", edgecolor="black")
    axes[1, 0].set_title("Max Heart Rate by Target")
    axes[1, 0].set_xlabel("Max Heart Rate")
    axes[1, 0].legend()

    # ST Depression vs Target
    for label, color in [(0, "#2ecc71"), (1, "#e74c3c")]:
        subset = cleveland[cleveland[TARGET_COLUMN] == label]
        axes[1, 1].hist(subset["st_depression"], bins=20, alpha=0.6, color=color,
                        label=f"{'Absence' if label == 0 else 'Presence'}", edgecolor="black")
    axes[1, 1].set_title("ST Depression by Target")
    axes[1, 1].set_xlabel("ST Depression")
    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()

## Section 14 — Categorical Features vs Target

In [ ]:
if cleveland is not None and TARGET_COLUMN in cleveland.columns:
    cat_plot_cols = ["sex", "chest_pain_type", "exercise_angina"]
    cat_plot_cols = [c for c in cat_plot_cols if c in cleveland.columns]
    n = len(cat_plot_cols)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, cat_plot_cols):
        ct = pd.crosstab(cleveland[col], cleveland[TARGET_COLUMN])
        ct.plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"])
        ax.set_title(f"{col} vs Target")
        ax.set_xlabel(col)
        ax.set_ylabel("Count")
        ax.legend(["Absence", "Presence"])
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    plt.tight_layout()
    plt.show()

## Section 15 — Key Observations

Document any observations about data quality here based on the analysis above.

In [ ]:
if cleveland is not None:
    print_dataset_summary(cleveland, "Cleveland Heart Disease")

## Section 16 — Framingham Dataset (if available)

In [ ]:
try:
    framingham = load_framingham_dataset()
    print(f"Framingham dataset loaded: {framingham.shape[0]} rows x {framingham.shape[1]} columns")
    print(f"\nColumns: {list(framingham.columns)}")
    print(framingham.head())
    print(framingham.dtypes)
except FileNotFoundError as e:
    print(f"Framingham dataset not found: {e}")
    print("Please place framingham.csv in data/raw/ and rerun this notebook.")
    framingham = None